# AIC System — Notebook 05: Run Queries, Export Submissions & BTC Evaluation (Combined)

**Supports all 3 AIC task types:** KIS (Dạng 1) · Q&A (Dạng 2) · TRAKE (Dạng 3)

## Notebook Features
1. **Pipeline Execution:** Runs multimodal search (CLIP + FAISS + OCR + VLM + Temporal Reranking).
2. **Submission Generation:** Generates `submission_kis.csv`, `submission_qa.csv`, `submission_trake.csv` for competition upload.
3. **BTC Offline Evaluation:** Automatically computes **BTC Final Score**, `Recall@1..100`, and `MRR` if Ground Truth is available.

In [ ]:
# ============================================================
# CELL 1: Setup Environment & Dependencies (Kaggle)
# ============================================================
import os, sys, subprocess
from pathlib import Path

REPO_DIR = Path("/kaggle/working/AIC_System")
if REPO_DIR.exists():
    sys.path.insert(0, str(REPO_DIR))
else:
    sys.path.insert(0, ".")

print("✅ Python path set up.")

In [ ]:
# ============================================================
# CELL 2: Configure Paths & Options
# ============================================================
from pathlib import Path

INDEX_DATASET     = "aic-hcmc-indexes"                  # Kaggle dataset slug for index files
AIC_DATA_DATASET  = "aic-hcmc-data"                     # Keyframe images
OCR_DATASET       = "extrac-ocr/ocr"                   # Extracted OCR JSON files
ENABLE_VLM        = True                                # Set False for fast CLIP-only mode

INDEX_DIR         = Path(f"/kaggle/input/{INDEX_DATASET}") if Path(f"/kaggle/input/{INDEX_DATASET}").exists() else Path("indexes")
KEYFRAME_ROOT     = Path(f"/kaggle/input/{AIC_DATA_DATASET}/keyframes") if Path(f"/kaggle/input/{AIC_DATA_DATASET}/keyframes").exists() else Path("datasets/keyframes")
OCR_DIR           = Path(f"/kaggle/input/{OCR_DATASET}") if Path(f"/kaggle/input/{OCR_DATASET}").exists() else Path("datasets/ocr")
QUERY_FILE        = Path("datasets/queries/sample_queries.json") if Path("datasets/queries/sample_queries.json").exists() else Path("/kaggle/input/aic-queries/queries.json")
GT_FILE           = Path("datasets/groundtruth/groundtruth_all.json")  # Optional Ground Truth
OUTPUT_DIR        = Path("/kaggle/working/submission")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Verify index files
required = ["faiss_visual.index", "keyframe_master.parquet"]
all_ok = True
for fname in required:
    fpath = INDEX_DIR / fname
    if fpath.exists():
        print(f"  ✅ Found {fname} ({fpath.stat().st_size / 1024 / 1024:.1f} MB)")
    else:
        print(f"  ⚠️ WARNING: {fname} not found in {INDEX_DIR}")
        all_ok = False

print(f"\n📁 Output dir: {OUTPUT_DIR}")
print(f"🖼️  Keyframe root: {KEYFRAME_ROOT}")
print(f"📝 OCR dir: {OCR_DIR} (exists={OCR_DIR.exists()})")

In [ ]:
# ============================================================
# CELL 3: Load Queries & Ground Truth
# ============================================================
import json

queries = []
if QUERY_FILE.exists():
    with open(QUERY_FILE, encoding="utf-8") as f:
        queries = json.load(f)
    print(f"✅ Loaded {len(queries)} queries from {QUERY_FILE}")
else:
    print(f"❌ Query file not found at {QUERY_FILE}")

ground_truth = {}
if GT_FILE.exists():
    with open(GT_FILE, encoding="utf-8") as f:
        ground_truth = json.load(f)
    print(f"⭐ Loaded ground truth for {len(ground_truth)} queries from {GT_FILE}")
else:
    print(f"ℹ️  No ground truth file found at {GT_FILE}. Evaluation step will be skipped.")

In [ ]:
# ============================================================
# CELL 4: Initialize RetrievalPipeline
# ============================================================
import time
from src.pipeline.retrieval_pipeline import RetrievalPipeline

print("⏳ Initializing RetrievalPipeline...")
t0 = time.time()

pipeline = RetrievalPipeline.from_index_dir(
    index_dir=str(INDEX_DIR),
    clip_model="ViT-B-32",
    clip_pretrained="openai",
    keyframe_image_root=str(KEYFRAME_ROOT),
    enable_vlm=ENABLE_VLM,
    vlm_model="Qwen/Qwen2.5-VL-7B-Instruct",
    vlm_load_in_4bit=True,
    ocr_dir=str(OCR_DIR) if OCR_DIR.exists() else None,
    top_k_retrieval=100,
    top_k_fusion=50,
    rrf_k=60,
)

print(f"✅ Pipeline ready in {time.time() - t0:.1f}s")

In [ ]:
# ============================================================
# CELL 5: Run Queries, Format Submissions & Compute Metrics
# ============================================================
from src.evaluation.submission_formatter import SubmissionFormatter
from src.evaluation.evaluator import Evaluator
from src.reasoning.query_classifier import QueryClassifier
from src.common.enums import QueryType

formatter  = SubmissionFormatter(output_dir=str(OUTPUT_DIR))
evaluator  = Evaluator()  # Default computes BTC_Final_Score and Recall@1, 5, 20, 50, 100
classifier = QueryClassifier()

all_results = []
errors = []

print(f"{'ID':<20} {'Type':<12} {'Video':<12} {'Frame':>7} {'pts(s)':>8} {'Score':>7} {'Latency':>8}")
print("-" * 85)

t_total = time.time()

for query_dict in queries:
    qid   = str(query_dict.get("query_id", "?"))
    qtype = classifier.classify(query_dict)
    t_q   = time.time()

    try:
        evidence = pipeline.run(query_dict, query_id=qid)
    except Exception as e:
        print(f"  ❌ [{qid}] ERROR: {e}")
        errors.append({"query_id": qid, "error": str(e)})
        continue

    if evidence is None:
        print(f"  ⚠️  [{qid}] No result returned")
        continue

    elapsed = time.time() - t_q

    # ── Route to Submission Formatter ─────────────────────────
    if qtype == QueryType.TEXTUAL_KIS:
        formatter.add_kis(qid, evidence)
    elif qtype == QueryType.QA:
        answer = evidence.metadata.get("answer", "")
        formatter.add_qa(qid, evidence, answer=str(answer))
    elif qtype == QueryType.TRAKE:
        trake_sub = evidence.metadata.get("trake_submission")
        if trake_sub is not None:
            event_frame_idxs = {ev.event_id: ev.frame_idx for ev in trake_sub.events}
            formatter.add_trake(qid, trake_sub.video_id, event_frame_idxs)
        else:
            fidx = evidence.frame_idx if evidence.frame_idx > 0 else 1
            formatter.add_trake(qid, evidence.video_id, {1: fidx, 2: fidx + 5, 3: fidx + 10})

    # ── Add to Evaluator if GT available ─────────────────────
    if qid in ground_truth:
        gt_info = ground_truth[qid]
        evaluator.add_result(
            query_id=qid,
            predicted_video=evidence.video_id,
            predicted_frame_idx=evidence.frame_idx,
            gt_video=gt_info.get("video_id", ""),
            gt_frame_idx=gt_info.get("frame_idx", 0),
            latency=elapsed,
        )

    all_results.append({
        "query_id": qid,
        "type":     qtype.value,
        "video_id": evidence.video_id,
        "frame_idx": evidence.frame_idx,
        "pts_time": round(evidence.pts_time, 2),
        "score":    round(evidence.confidence, 4),
        "answer":   evidence.metadata.get("answer", ""),
        "latency_s": round(elapsed, 2),
    })

    print(f"  ✅ {qid:<20} {qtype.value:<12} {evidence.video_id:<12} "
          f"{evidence.frame_idx:>8} {evidence.pts_time:>8.2f}s "
          f"{evidence.confidence:>7.4f} {elapsed:>7.2f}s")

print("-" * 85)
print(f"Total: {len(all_results)}/{len(queries)} queries processed, "
      f"{len(errors)} errors in {time.time()-t_total:.1f}s")

In [ ]:
# ============================================================
# CELL 6: Export Submission Files & Offline Report
# ============================================================
import pandas as pd

paths = formatter.save_all()

print("\n📦 Generated Submission Files:")
for key, p in paths.items():
    if p.exists():
        rows = pd.read_csv(p) if str(p).endswith(".csv") else None
        n = len(rows) if rows is not None else "-"
        print(f"  • {key.upper():<8} {p.name:<30} ({n} rows)")

if ground_truth:
    evaluator.print_report()
    report_path = OUTPUT_DIR / "eval_report.json"
    evaluator.save(str(report_path))
    print(f"⭐ Detailed evaluation report saved → {report_path}")

In [ ]:
# ============================================================
# CELL 7: Preview Query Results & Submissions
# ============================================================
df_results = pd.DataFrame(all_results)
display(df_results.head(10))

print("\n=== KIS Submission Preview ===")
kis_p = OUTPUT_DIR / "submission_kis.csv"
if kis_p.exists():
    display(pd.read_csv(kis_p).head(5))

print("\n=== Q&A Submission Preview ===")
qa_p = OUTPUT_DIR / "submission_qa.csv"
if qa_p.exists():
    display(pd.read_csv(qa_p).head(5))

print("\n=== TRAKE Submission Preview ===")
trake_p = OUTPUT_DIR / "submission_trake.csv"
if trake_p.exists():
    display(pd.read_csv(trake_p).head(5))